# GWAS Atlas — API & bulk

[GWAS Atlas](https://atlas.ctglab.nl/) (Watanabe et al., Nat Genet 2019)
publishes a (gene × study) MAGMA p-value matrix across ~4,000 GWAS
summary statistics. The Atlas has no structured REST API — all
downloads go through a Laravel CSRF-protected form, which `bioDB`
handles transparently.

| Mode | Functions |
|---|---|
| **Per-trait lookup** | `query_trait`, `list_traits` (in-memory index over the metadata TSV) |
| **Bulk download** | `download_metadata`, `download_magma_p`, `load_metadata`, `load_magma_p`, `melt_magma_p` |

In [1]:
from biodb import gwas_atlas

## 1. Per-trait lookup — Alzheimer's example

First call downloads the metadata TSV (a few MB) into
`~/.cache/biodb/gwas_atlas/`. Subsequent calls are instant — the
metadata is cached in module memory.

In [2]:
ad = gwas_atlas.query_trait("Alzheimer")
print(ad.shape)
ad[["id", "Trait", "Year", "N", "Domain", "PMID"]].head()

(9, 29)


,id,Trait,Year,N,Domain,PMID
33,34,Alzheimer's disease,2013,54162,Neurological,24162737
1080,1081,Alzheimer's disease,2008,1489,Neurological,17998437
3634,3635,Illnesses of father: Alzheimer's disease/dementia,2019,355137,Environment,31427789
3645,3646,Illnesses of mother: Alzheimer's disease/dementia,2019,367939,Environment,31427789
4087,4088,Paternal history of Alzheimer's disease,2018,269279,Environment,29777097


Lookup by numeric `id` is auto-detected from the input shape:

In [3]:
gwas_atlas.query_trait(1)[["id", "Trait", "Year", "PMID"]].head()

,id,Trait,Year,PMID
0,1,Attention deficit hyperactivity disorder,2010,20732625


Force the lookup column with `column=`:

In [4]:
gwas_atlas.query_trait("28067908", column="PMID")[["id", "Trait", "PMID"]].head()

,id,Trait,PMID
2028,2029,Crohn's Disease,28067908
2029,2030,Ulcerative colitis,28067908
2030,2031,Inflammatory Bowel Disease,28067908


## 2. Bulk — the (gene × study) MAGMA matrix

`download_magma_p` pulls the full p-value matrix (~hundreds of MB).
`melt_magma_p` reshapes wide-format `(gene × study)` to long-format
`(targetId, sourceId, score)`, matching what
`biodb.transform.create_gene_association_matrix` expects.

```python
magma = gwas_atlas.load_magma_p()         # wide (gene × study)
long  = gwas_atlas.melt_magma_p(magma)    # (targetId, sourceId, score)
```

In [5]:
gwas_atlas.list_traits().shape

(4756, 29)